In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

In [0]:
BRONZE_PATH = "abfss://bronze@pravdatalake.dfs.core.windows.net"
SILVER_PATH = "abfss://silver@pravdatalake.dfs.core.windows.net"
SILVER_TABLE_PATH = f"{SILVER_PATH}/ad_table"
SILVER_TABLE_NAME = "vehicle_sales.silver.ad_table"

In [0]:
ad_df = spark.read.format("csv")\
    .option("header", True)\
    .option("inferSchema", True)\
    .load(f"{BRONZE_PATH}/Ad_table")

In [0]:
ad_df.display()

In [0]:
print(f"bronze row count: {ad_df.count()}")

In [0]:
silver_ad = (
    ad_df
    .withColumn("Genmodel_ID", trim(col("Genmodel_ID")))
    .withColumn("Maker", trim(initcap(col("Maker"))))
    .withColumn("Genmodel", trim(col("Genmodel")))
    .withColumn("Adv_ID", trim(col("Adv_ID")))
    .withColumn("Adv_year", expr("try_cast(regexp_replace(Adv_year, '[^0-9]', '') as int)"))
    .withColumn("Adv_month", expr("try_cast(regexp_replace(Adv_month, '[^0-9]', '') as int)"))
    .withColumn("Color", trim(initcap(col("Color"))))
    .withColumn("Reg_year", expr("try_cast(regexp_replace(Reg_year, '[^0-9]', '') as int)"))
    .withColumn("Bodytype", trim(initcap(col("Bodytype"))))
    .withColumn("Runned_Miles", expr("try_cast(regexp_replace(Runned_Miles, '[^0-9.]', '') as double)"))
    .withColumn("Engine_size_l", expr("try_cast(regexp_replace(Engin_size, '[^0-9.]', '') as double)"))
    .withColumn("Gearbox", trim(initcap(col("Gearbox"))))
    .withColumn("Fuel_type", trim(initcap(col("Fuel_type"))))
    .withColumn("Price", expr("try_cast(regexp_replace(Price, '[^0-9.]', '') as double)"))
    .withColumn("Seat_num", expr("try_cast(regexp_replace(Seat_num, '[^0-9]', '') as int)"))
    .withColumn("Door_num", expr("try_cast(regexp_replace(Door_num, '[^0-9]', '') as int)"))
    .filter(col("Adv_ID").isNotNull())
    .filter(col("Genmodel_ID").isNotNull())
    .filter((col("Price").isNull()) | (col("Price") >= 0))
    .filter((col("Runned_Miles").isNull()) | (col("Runned_Miles") >= 0))
    .dropDuplicates(["Adv_ID"])
    .withColumn("silver_ingestion_timestamp", current_timestamp())
)

In [0]:
silver_ad.display()

####Data Quality checks

In [0]:
row_count = silver_ad.count()

In [0]:
null_key_count = silver_ad.filter(col("Adv_ID").isNull()).count()

In [0]:
duplicate_key_count = silver_ad.groupBy("Adv_ID").count().filter("count > 1").count()

In [0]:
negative_price_count = silver_ad.filter(col("Price") < 0).count()

In [0]:
print(f"silver row count: {row_count}")
print(f"null Adv_ID count: {null_key_count}")
print(f"duplicate Adv_ID count: {duplicate_key_count}")
print(f"negative Price count: {negative_price_count}")

In [0]:
assert null_key_count == 0, "Adv_ID should never be null in silver_ad"
assert duplicate_key_count == 0, "Adv_ID should be unique in silver_ad"
assert negative_price_count == 0, "Price should never be negative"

In [0]:
if DeltaTable.isDeltaTable(spark, SILVER_TABLE_PATH):
 
    silver_table = DeltaTable.forPath(spark, SILVER_TABLE_PATH)
 
    (silver_table.alias("t")
        .merge(silver_ad.alias("s"), "t.Adv_ID = s.Adv_ID")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
 
else:
 
    silver_ad.write \
        .format("delta") \
        .mode("overwrite") \
        .partitionBy("Adv_year") \
        .save(SILVER_TABLE_PATH)

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER_TABLE_NAME}
    USING DELTA
    LOCATION '{SILVER_TABLE_PATH}'
""")

In [0]:
spark.sql(f"OPTIMIZE {SILVER_TABLE_NAME} ZORDER BY (Genmodel_ID)")